# Reccomended Additions

This notebook is a planning checklist of model improvements to implement later for a more accurate and robust DiD analysis.

## 1) Data consistency and identification

### A. Use one UK academic-year mapping rule everywhere
- Choose one rule and keep it fixed across all models:
  - Start-year mapping: 2019/20 -> 2019
  - End-year mapping: 2019/20 -> 2020
- Re-run all baseline and robustness models after fixing this choice.

### B. Stronger pre-trends testing
- Keep event-study plots.
- Add a joint test that all pre-policy leads are zero.
- Document both visual and statistical pre-trend evidence.

### C. Placebo policy dates
- Run placebo DiD with fake treatment starts (for example 2018 or 2019).
- Expected result: no significant policy effect in fake pre-periods.

## 2) Alternative specifications

### A. COVID sensitivity
Estimate multiple variants:
1. Current model with Australia-specific COVID controls.
2. Drop 2020 only.
3. Drop both 2020 and 2021.
Then compare sign, magnitude, and significance of the DiD effect.

### B. Alternative outcomes
Run models using:
- log enrollments (current),
- level enrollments,
- enrollment share by discipline.
Consistency across outcomes increases confidence.

### C. Discipline-specific trends
- Add category-specific linear trends to absorb slow-moving discipline dynamics.
- Check whether core treatment estimates remain stable.

## 3) Inference and statistical reliability

### A. Small treated-group inference
- Standard errors can be fragile with one treated country.
- Add wild-cluster bootstrap or randomization inference style p-values as robustness checks.

### B. Multiple testing correction
- For many discipline-specific regressions, adjust p-values (for example Benjamini-Hochberg/FDR).
- Report both raw and adjusted significance.

### C. Confidence intervals and effect sizes
- Report coefficients in log points and approximate percentage effects.
- Always present 95% confidence intervals for key estimates.

## 4) Additional controls (if available)

Potential confounders to add annually:
- Higher education funding changes,
- labor market demand/unemployment indicators,
- migration or international student shocks.

Goal: reduce omitted-variable bias in policy effect estimates.

## 5) Suggested implementation order

1. Fix UK year mapping and rerun baseline.
2. Run pre-trends joint test and placebo dates.
3. Run COVID and outcome sensitivity checks.
4. Add stronger inference (bootstrap/randomization).
5. Add multiple-testing correction for discipline results.
6. Add extra controls where data is available.

## 6) Notes section

Use this section to track what has been implemented and what remains pending.

## To Do

- **Extend DiD panel start year from 2019 to 2016 across all REG notebooks.**
  AUS enrolment data is available from 2016. However, the UK enrolment data with
  correctly mapped AUS category keys only exists from 2019/20 onwards — the earlier
  UK HESA files (2016/17 – 2018/19) use a different subject classification where
  Architecture, Creative Arts, and Education are grouped under a single "Others"
  category (key=11), and Engineering maps to a different key entirely.

  **Pre-requisite:** reclassify and remap the 2016/17 – 2018/19 UK grouped files
  to the AUS category key taxonomy (keys 1–10). Once done, re-run the DiD panel
  construction in each REG notebook, changing the AUS filter from
  `arch_aus[arch_aus['year'] >= 2019]` to `arch_aus[arch_aus['year'] >= 2016]`.
  This would add 3 additional pre-treatment years, enabling a richer parallel
  trends test and improving statistical power (df rises from 4 to 7 per notebook).


## 7) Instrumental Variable Extension (future analysis)

### Should an IV be added to the current DiD?

**No — not in the current country-level DiD design.** The treatment is binary and
country-assigned (AUS received JRG, UK did not). There is no selection into treatment
at the unit level, so the standard IV motivation (endogeneity of treatment assignment)
does not apply. Any country-level instrument would also be perfectly collinear with the
`treated` dummy. With N=18 and df=7, there are also insufficient degrees of freedom to
support a two-stage regression.

### Where an IV would add value — a pooled cross-discipline design

The more interesting endogeneity problem is not *which country* got JRG but *how much
fees changed* by discipline. JRG moved disciplines very differently:

| Discipline | Fee change | Classification |
|------------|------------|----------------|
| Education | −40% | Priority |
| Engineering & Related Tech | −16% | Priority |
| Architecture & Building | varies (increase) | Non-priority |
| Creative Arts | +45% | Non-priority |

If disciplines were pooled into a single panel (`discipline × country × year`),
an IV approach becomes viable:

- **Endogenous variable:** actual student contribution change (% or $) for each AUS
  discipline post-2021
- **Instrument:** JRG priority classification (binary) × post-2021

**Why this works as an IV:**
- *Relevance:* Priority classification is strongly correlated with the fee change —
  it is essentially what determined the direction and magnitude of fee movements
- *Exclusion restriction:* The administrative classification (set before JRG) plausibly
  has no direct effect on enrollment except through the fee channel

This would estimate a **LATE** (Local Average Treatment Effect) — the enrollment
elasticity for disciplines whose fees were moved by JRG's priority designation — which
is the directly policy-relevant parameter (fee elasticity of enrollment).

### Caveat on the exclusion restriction

The exclusion restriction is debatable. Priority classification did not *only* change
fees — it also changed Commonwealth contributions and carried a government signal about
the perceived value of a field (potentially affecting student perception of career
prospects). If 'priority' status has its own signalling effect on enrollment beyond
fees, the exclusion restriction fails.

### Implementation when ready

1. Pool all discipline-level panels into a single `discipline × country × year` dataset
2. Add `student_contribution_pct_change` as the endogenous treatment variable
3. Construct instrument: `priority_flag × post_2021` (where `priority_flag` is the
   pre-determined JRG classification for each AUS discipline)
4. Run 2SLS with discipline FE, country FE, and year FE
5. Report first-stage F-statistic (should exceed 10 for instrument relevance)
6. Compare 2SLS estimate to OLS DiD to assess attenuation/amplification bias
